In [14]:
import pandas as pd
import numpy as np
import tensorflow as tf
import random

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    fbeta_score
)
from imblearn.over_sampling import SMOTE

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, LSTM,
    Dense, Add, Attention, Lambda
)
from tensorflow.keras.optimizers import Nadam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

In [15]:
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [16]:
df_train = pd.read_csv("F:\\Dataset\\ECG Dataset\\mitbih_train.csv", header=None)
df_test = pd.read_csv("F:\\Dataset\\ECG Dataset\\mitbih_test.csv", header=None)

X_train = df_train.iloc[:, :-1].values
y_train = df_train.iloc[:, -1].values

X_test = df_test.iloc[:, :-1].values
y_test = df_test.iloc[:, -1].values

In [17]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [18]:
scaler = StandardScaler()
X_train_smote = scaler.fit_transform(X_train_smote)
X_test = scaler.transform(X_test)

In [19]:
X_train_smote = X_train_smote.reshape((-1, 187, 1))
X_test = X_test.reshape((-1, 187, 1))

In [20]:
y_train_smote = to_categorical(y_train_smote)
y_test_encoded = to_categorical(y_test)

In [21]:
# Dynamic Temporal Positional Encoding
def add_positional_encoding(x):
    T = tf.shape(x)[1]
    D = tf.shape(x)[2]

    pos = tf.range(T, dtype=tf.float32)[:, tf.newaxis]
    i = tf.range(D, dtype=tf.float32)[tf.newaxis, :]

    angle_rates = 1.0 / tf.pow(10000.0, (2*(i//2)) / tf.cast(D, tf.float32))
    angle_rads = pos * angle_rates

    sines = tf.sin(angle_rads[:, 0::2])
    cosines = tf.cos(angle_rads[:, 1::2])

    pe = tf.concat([sines, cosines], axis=-1)
    pe = tf.expand_dims(pe, axis=0)

    return x + pe


# Model
input_layer = Input(shape=(187, 1))

x = Conv1D(32, 3, activation='relu', padding='same')(input_layer)
x = MaxPooling1D(2)(x)

x = Conv1D(64, 3, activation='relu', padding='same')(x)
x = MaxPooling1D(2)(x)

# Add Positional Encoding
x = Lambda(add_positional_encoding)(x)

# Proper Keras Attention (2-input)
Q = Dense(64)(x)
V = Dense(64)(x)

attention_output = Attention()([Q, V])
x = Add()([x, attention_output])

# LSTM
x = LSTM(64)(x)

x = Dense(32, activation='relu')(x)

output_layer = Dense(5, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output_layer)

In [22]:
model.compile(
    optimizer=Nadam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [23]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train_smote,
    y_train_smote,
    epochs=20,
    batch_size=64,
    validation_data=(X_test, y_test_encoded),
    callbacks=[early_stop]
)

Epoch 1/20
5662/5662 ━━━━━━━━━━━━━━━━━━━━ 204s 35ms/step - accuracy: 0.9134 - loss: 0.2443 - val_accuracy: 0.9294 - val_loss: 0.1985
Epoch 2/20
5662/5662 ━━━━━━━━━━━━━━━━━━━━ 208s 37ms/step - accuracy: 0.9651 - loss: 0.1002 - val_accuracy: 0.9514 - val_loss: 0.1476
Epoch 3/20
5662/5662 ━━━━━━━━━━━━━━━━━━━━ 237s 42ms/step - accuracy: 0.9744 - loss: 0.0751 - val_accuracy: 0.9522 - val_loss: 0.1472
Epoch 4/20
5662/5662 ━━━━━━━━━━━━━━━━━━━━ 260s 46ms/step - accuracy: 0.9776 - loss: 0.0648 - val_accuracy: 0.9567 - val_loss: 0.1366
Epoch 5/20
5662/5662 ━━━━━━━━━━━━━━━━━━━━ 248s 44ms/step - accuracy: 0.9803 - loss: 0.0577 - val_accuracy: 0.9720 - val_loss: 0.1025
Epoch 6/20
5662/5662 ━━━━━━━━━━━━━━━━━━━━ 237s 42ms/step - accuracy: 0.9820 - loss: 0.0532 - val_accuracy: 0.9702 - val_loss: 0.1014
Epoch 7/20
5662/5662 ━━━━━━━━━━━━━━━━━━━━ 289s 51ms/step - accuracy: 0.9834 - loss: 0.0489 - val_accuracy: 0.9631 - val_loss: 0.1205
Epoch 8/20
5662/5662 ━━━━━━━━━━━━━━━━━━━━ 248s 44ms/step - accuracy: 

In [24]:
test_loss, test_acc = model.evaluate(X_test, y_test_encoded)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")

685/685 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9702 - loss: 0.1014

Final Test Accuracy: 97.02%


In [25]:
# Predictions
pred_probs = model.predict(X_test)
y_pred = np.argmax(pred_probs, axis=1)
y_true = np.argmax(y_test_encoded, axis=1)

cm = confusion_matrix(y_true, y_pred)
num_classes = cm.shape[0]

accuracy = np.sum(np.diag(cm)) / np.sum(cm)

precision_list = []
recall_list = []
specificity_list = []
f1_list = []
npv_list = []
fpr_list = []
f2_list = []
mcc_list = []

for i in range(num_classes):
    TP = cm[i, i]
    FN = np.sum(cm[i, :]) - TP
    FP = np.sum(cm[:, i]) - TP
    TN = np.sum(cm) - (TP + FP + FN)

    precision = TP / (TP + FP) if (TP + FP) != 0 else 0
    recall = TP / (TP + FN) if (TP + FN) != 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) != 0 else 0
    npv = TN / (TN + FN) if (TN + FN) != 0 else 0
    fpr = FP / (FP + TN) if (FP + TN) != 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
    f2 = fbeta_score(y_true == i, y_pred == i, beta=2)

    denom = np.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
    mcc = ((TP*TN)-(FP*FN))/denom if denom != 0 else 0

    precision_list.append(precision)
    recall_list.append(recall)
    specificity_list.append(specificity)
    f1_list.append(f1)
    npv_list.append(npv)
    fpr_list.append(fpr)
    f2_list.append(f2)
    mcc_list.append(mcc)

results = {
    "Model": "EDATNet",
    "Acc (%)": accuracy * 100,
    "Pre (%)": np.mean(precision_list) * 100,
    "Rec (%)": np.mean(recall_list) * 100,
    "Spe (%)": np.mean(specificity_list) * 100,
    "F1 (%)": np.mean(f1_list) * 100,
    "AUC (%)": roc_auc_score(y_test_encoded, pred_probs, multi_class='ovr') * 100,
    "MCC (%)": np.mean(mcc_list) * 100,
    "NPV (%)": np.mean(npv_list) * 100,
    "FPR (%)": np.mean(fpr_list) * 100,
    "F2 (%)": np.mean(f2_list) * 100
}

results_df = pd.DataFrame([results])
print("\nFinal Evaluation Metrics:\n")
print(results_df.round(2))

685/685 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step

Final Evaluation Metrics:

     Model  Acc (%)  Pre (%)  Rec (%)  Spe (%)  F1 (%)  AUC (%)  MCC (%)  \
0  EDATNet    97.02    82.56    92.28    98.69   86.52    98.91    85.17   

   NPV (%)  FPR (%)  F2 (%)  
0     97.7     1.31   89.69  
